In [1]:
import pandas as pd

from llm_audit import BASE_DIR


EN_PATH = BASE_DIR / "eval" / "data" / "boot" / "psyc_open_en_meta.csv"
ZH_PATH = BASE_DIR / "eval" / "data" / "boot" / "psyc_open_zh_meta.csv"


def fmt(mean, lo, hi, dec=2):
    """Format mean [lo, hi] compactly."""
    f = f".{dec}f"
    return f"{mean:{f}} [{lo:{f}},{hi:{f}}]"


def load(path):
    df = pd.read_csv(path)
    df = df.set_index("Model")
    return df


def make_row(row_en, row_zh):
    cells = []
    for row in (row_en, row_zh):
        for metric in ("TPR", "FPR", "TPR-FPR"):
            cells.append(
                fmt(
                    row[f"{metric}_mean"],
                    row[f"{metric}_ci_lo"],
                    row[f"{metric}_ci_up"],
                )
            )
    return cells


en = load(EN_PATH)
zh = load(ZH_PATH)

models = en.index.intersection(zh.index)

# Header
cols_en = ["EN TPR", "EN FPR", "EN TPR-FPR"]
cols_zh = ["ZH TPR", "ZH FPR", "ZH TPR-FPR"]
header = ["Model"] + cols_en + cols_zh
sep = [":---"] + ["---:"] * 6

lines = []
lines.append("|" + "|".join(header) + "|")
lines.append("|" + "|".join(sep) + "|")

for model in models:
    short = model.split("/")[-1]  # strip org prefix to save chars
    row = [short] + make_row(en.loc[model], zh.loc[model])
    lines.append("|" + "|".join(row) + "|")

table = "\n".join(lines)
print(table)

|Model|EN TPR|EN FPR|EN TPR-FPR|ZH TPR|ZH FPR|ZH TPR-FPR|
|:---|---:|---:|---:|---:|---:|---:|
|Qwen3-30B-A3B-Instruct-2507|0.67 [0.57,0.77]|0.19 [0.12,0.26]|0.48 [0.36,0.61]|0.59 [0.27,0.89]|0.11 [0.00,0.26]|0.49 [0.14,0.84]|
|QVikhr-3-8B-Instruction|0.67 [0.57,0.77]|0.19 [0.12,0.26]|0.49 [0.36,0.60]|0.60 [0.27,0.90]|0.10 [0.00,0.25]|0.50 [0.14,0.82]|
|GigaChat-20B-A3B-instruct|0.67 [0.57,0.77]|0.19 [0.12,0.26]|0.48 [0.36,0.61]|0.61 [0.29,0.91]|0.10 [0.00,0.27]|0.50 [0.14,0.84]|
|Olmo-3.1-32B-Instruct|0.67 [0.57,0.77]|0.19 [0.12,0.26]|0.48 [0.36,0.61]|0.60 [0.27,0.90]|0.11 [0.00,0.27]|0.49 [0.15,0.83]|
|claude-haiku-4.5|0.67 [0.57,0.77]|0.19 [0.12,0.26]|0.48 [0.36,0.60]|0.60 [0.25,0.89]|0.11 [0.00,0.27]|0.49 [0.13,0.83]|
|deepseek-v3.2|0.67 [0.57,0.77]|0.19 [0.12,0.26]|0.48 [0.36,0.60]|0.60 [0.29,0.90]|0.10 [0.00,0.26]|0.49 [0.13,0.80]|
|gemini-3-flash-preview|0.67 [0.57,0.77]|0.19 [0.12,0.26]|0.48 [0.36,0.60]|0.60 [0.27,0.89]|0.11 [0.00,0.27]|0.49 [0.15,0.83]|
|mistral-large-2512|0.6